In [ ]:
# Title:
# achieve a attention mechanism

import torch
import torch.nn as nn
# vocab
input_vocab = {'<SOS>': 0, "let's": 1, 'to': 2, 'go': 3, '<EOS>': 4}
output_vocab = {'<SOS>': 0, 'ir': 1, 'vamos': 2, 'y': 3, '<EOS>': 4}

# word embedding
class Embedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Embedding, self).__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, embedding_dim))

    def forward(self, x):
        return self.embedding[x]

# positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super(PositionalEncoding, self).__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.positional_encoding = nn.Parameter(torch.randn(input_dim, embedding_dim))

    def forward(self, x):
        return x + self.positional_encoding[:x.size(0), :]

# linear: xW + B 
class Linear(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Linear, self).__init__()
        # initialize weights and bias
        self.W = nn.Parameter(torch.randn(input_dim, output_dim))
        self.B = nn.Parameter(torch.randn(output_dim))

    def forward(self, x):
        return x @ self.W + self.B

# attn
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim, isdecoder=False):
        super(SelfAttention, self).__init__()
        self.embedding_dim = embedding_dim
        self.isdecoder = isdecoder
        self.W_q = nn.Parameter(torch.randn(embedding_dim, embedding_dim))
        self.W_k = nn.Parameter(torch.randn(embedding_dim, embedding_dim))
        self.W_v = nn.Parameter(torch.randn(embedding_dim, embedding_dim))

    def forward(self, x):
        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v     

        unscaled_dot_product_similarity = Q @ (K.transpose(0, 1))
        scaled_dot_product_similarity = unscaled_dot_product_similarity / (self.embedding_dim ** 0.5)
        if self.isdecoder:
            mask = torch.triu(torch.ones_like(scaled_dot_product_similarity), diagonal=1).bool()
            scaled_dot_product_similarity = scaled_dot_product_similarity.masked_fill(mask, float('-inf'))
        attn_weights = torch.softmax(scaled_dot_product_similarity, dim=1)
        attn_score = attn_weights @ V
        return attn_score

class CrossAttention(nn.Module):
    def __init__(self, embedding_dim):
        super(CrossAttention, self).__init__()
        self.embedding_dim = embedding_dim
        self.W_q = nn.Parameter(torch.randn(embedding_dim, embedding_dim))
        self.W_k = nn.Parameter(torch.randn(embedding_dim, embedding_dim))
        self.W_v = nn.Parameter(torch.randn(embedding_dim, embedding_dim))

    def forward(self, decoder_provide, encoder_provide):
        Q = decoder_provide @ self.W_q
        K = encoder_provide @ self.W_k
        V = encoder_provide @ self.W_v

        unscaled_dot_product_similarity = Q @ (K.transpose(0, 1))
        scaled_dot_product_similarity = unscaled_dot_product_similarity / (self.embedding_dim ** 0.5)
        attn_weights = torch.softmax(scaled_dot_product_similarity, dim=1)
        attn_score = attn_weights @ V
        return attn_score

class layer_norm(nn.Module):
    def __init__(self, embedding_dim, eps=1e-6):
        super(layer_norm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(embedding_dim))
        self.beta = nn.Parameter(torch.zeros(embedding_dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        normalized_x = (x - mean) / (std + self.eps)
        return self.gamma * normalized_x + self.beta

class ffn(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super(ffn, self).__init__()
        self.linear1 = Linear(embedding_dim, hidden_dim)
        self.linear2 = Linear(hidden_dim, embedding_dim)

    def forward(self, x):
        x = torch.relu(self.linear1(x))
        x = self.linear2(x)
        return x

class encoder_block(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(encoder_block, self).__init__()
        self.self_attention = SelfAttention(embedding_dim=embedding_dim, isdecoder=False)
        self.layer_norm = layer_norm(embedding_dim=embedding_dim)
        self.layer_norm_2 = layer_norm(embedding_dim=embedding_dim)
        self.ffn = ffn(embedding_dim=embedding_dim, hidden_dim=embedding_dim * 4)

    def forward(self, x):
        attn_output = self.self_attention.forward(x)
        attn_encoded = attn_output + x
        normalized_output = self.layer_norm(attn_encoded)
        ffn_output = self.ffn(normalized_output)
        residual_output = ffn_output + normalized_output
        residual_output = self.layer_norm_2(residual_output)  
        return residual_output


class decoder_block(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(decoder_block, self).__init__()
        self.self_attention = SelfAttention(embedding_dim=embedding_dim, isdecoder=True)
        self.cross_attention = CrossAttention(embedding_dim=embedding_dim)
        self.layer_norm = layer_norm(embedding_dim=embedding_dim)
        self.layer_norm_2 = layer_norm(embedding_dim=embedding_dim)  
        self.layer_norm_3 = layer_norm(embedding_dim=embedding_dim)  
        self.ffn = ffn(embedding_dim=embedding_dim, hidden_dim=embedding_dim * 4)
        self.fc = Linear(input_dim=embedding_dim, output_dim=vocab_size)

    def forward(self, x, encoder_output):
        attn_output = self.self_attention.forward(x)
        attn_encoded = attn_output + x
        normalized_output = self.layer_norm(attn_encoded)

        cross_attn_output = self.cross_attention.forward(
            normalized_output, encoder_output
        )
        re_output = cross_attn_output + normalized_output
        re_output = self.layer_norm_2(re_output)  

        ffn_output = self.ffn(re_output)
        residual_output = ffn_output + re_output
        residual_output = self.layer_norm_3(residual_output)  

        return residual_output

class Transformer(nn.Module):
    def __init__(self, input_vocab_size, output_vocab_size, embedding_dim, block_num):
        super(Transformer, self).__init__()
        embedding_dim = embedding_dim
        self.embedding = Embedding(input_vocab_size, embedding_dim)
        self.positional_encoding = PositionalEncoding(input_dim=128, embedding_dim=embedding_dim)

        self.decoder_embedding = Embedding(output_vocab_size, embedding_dim)
        self.decoder_positional_encoding = PositionalEncoding(input_dim=128, embedding_dim=embedding_dim)

        self.encoder = nn.ModuleList(
            [encoder_block(input_vocab_size, embedding_dim) for _ in range(block_num)]
        )
        self.decoder = nn.ModuleList(
            [decoder_block(output_vocab_size, embedding_dim) for _ in range(block_num)]
        )

    def forward(self, src, tgt):
        embedded = self.embedding.forward(src)
        pos_encoded = self.positional_encoding.forward(embedded)

        encoder_output = pos_encoded
        for block in self.encoder:
            encoder_output = block(encoder_output)

        decoder_embedded = self.decoder_embedding.forward(tgt)
        decoder_pos_encoded = self.decoder_positional_encoding.forward(decoder_embedded)

        decoder_output = decoder_pos_encoded
        for block in self.decoder:
            decoder_output = block(decoder_output, encoder_output)

        decoder_output = self.decoder[-1].fc(decoder_output)
        return decoder_output

model = Transformer(input_vocab_size=len(input_vocab), output_vocab_size=len(output_vocab), embedding_dim=64, block_num=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
src = torch.tensor([input_vocab["<SOS>"], input_vocab["let's"], input_vocab["go"], input_vocab["<EOS>"]])
tgt = torch.tensor([output_vocab["<SOS>"], output_vocab["vamos"], output_vocab["<EOS>"]])

for epoch in range(100):
    optimizer.zero_grad()
    decoder_input = tgt[:-1]
    target = tgt[1:]
    output = model(src, decoder_input)
    loss = criterion(output.view(-1, output.size(-1)), target.view(-1))
    loss.backward()
    # print the loss every 10 epochs and result
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")
    print("Predicted output:", [list(output_vocab.keys())[list(output_vocab.values()).index(idx.item())] for idx in torch.argmax(output, dim=-1)])
    optimizer.step()

# Test the model
model.eval()
test_src = torch.tensor([input_vocab["<SOS>"], input_vocab["let's"], input_vocab["go"], input_vocab["<EOS>"]])
test_tgt = torch.tensor([output_vocab["<SOS>"], output_vocab["vamos"], output_vocab["<EOS>"]])  
predicted_output = model(test_src, test_tgt[:-1])
predicted_indices = torch.argmax(predicted_output, dim=-1)
predicted_words = [list(output_vocab.keys())[list(output_vocab.values()).index(idx.item())] for idx in predicted_indices]
print("Predicted translation:", predicted_words)